In [ ]:
!pip install -q unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True
Tesla T4


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 386
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
from google.colab import files
uploaded = files.upload()  # select training_data.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="final_train.jsonl", split="train")
print(dataset)
print(dataset[0])

Saving final_train.jsonl to final_train (1).jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 176
})
{'system': 'You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.', 'user': 'CRM: price $24.76, RSI 49, company initiated its first-ever quarterly dividend, 5d momentum 0.6%.', 'assistant': '{"action": "buy", "confidence": 0.63, "rationale": "Initiating a dividend signals management\'s confidence in sustainable free cash flow and typically broadens the shareholder base to income-focused investors."}'}


In [ ]:
def format_example(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)

print(dataset[0]["text"])
print("---")

lengths = [len(tokenizer(ex["text"])["input_ids"]) for ex in dataset]
print("Token lengths — min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths)/len(lengths))

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

<|im_start|>system
You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.<|im_end|>
<|im_start|>user
CRM: price $24.76, RSI 49, company initiated its first-ever quarterly dividend, 5d momentum 0.6%.<|im_end|>
<|im_start|>assistant
{"action": "buy", "confidence": 0.63, "rationale": "Initiating a dividend signals management's confidence in sustainable free cash flow and typically broadens the shareholder base to income-focused investors."}<|im_end|>

---
Token lengths — min: 111 max: 160 avg: 137.33522727272728


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/176 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/176 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 176 | Num Epochs = 2 | Total steps = 44
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.369083
2,2.573341
3,2.435688
4,2.319747
5,2.126306
6,1.904411
7,1.829675
8,1.739167
9,1.839941
10,1.723996


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-44/tokenizer_config.json.


In [ ]:
model.save_pretrained("qwen_trading_lora")
tokenizer.save_pretrained("qwen_trading_lora")

!zip -r qwen_trading_lora.zip qwen_trading_lora
from google.colab import files
files.download("qwen_trading_lora.zip")

Unsloth: Restored added_tokens_decoder metadata in qwen_trading_lora/tokenizer_config.json.


  adding: qwen_trading_lora/ (stored 0%)
  adding: qwen_trading_lora/adapter_model.safetensors (deflated 8%)
  adding: qwen_trading_lora/tokenizer_config.json (deflated 89%)
  adding: qwen_trading_lora/adapter_config.json (deflated 59%)
  adding: qwen_trading_lora/chat_template.jinja (deflated 71%)
  adding: qwen_trading_lora/README.md (deflated 65%)
  adding: qwen_trading_lora/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": "You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale."},
    {"role": "user", "content": "COIN: price $210.40, RSI 74, 5d momentum +11.8%, price up sharply after Bitcoin rallied 9% this week, volume 2x average."},
]

inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"action": "sell", "confidence": 0.63, "rationale": "RSI of 74 indicates the move is driven by short-term participants rather than fundamental factors, and sharp price repositioning after a broader crypto rally raises reversal risk."}


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving final_eval.jsonl to final_eval.jsonl


In [ ]:
import json, re

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

eval_data = load_jsonl("final_eval.jsonl")  # upload this the same way as before

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def run_eval(model, tokenizer, eval_data, label=""):
    FastLanguageModel.for_inference(model)
    results = []
    for ex in eval_data:
        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
        ]
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=False)
        raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = extract_json(raw)
        true_label = json.loads(ex["assistant"])
        results.append({
            "user": ex["user"],
            "raw_output": raw,
            "parsed": parsed,
            "valid_json": parsed is not None,
            "true_action": true_label["action"],
            "pred_action": parsed["action"] if parsed else None,
            "correct": parsed is not None and parsed.get("action") == true_label["action"],
            "pred_confidence": parsed.get("confidence") if parsed else None,
        })
    return results

ft_results = run_eval(model, tokenizer, eval_data, label="fine-tuned")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [ ]:
def summarize(results, label=""):
    n = len(results)
    valid = [r for r in results if r["valid_json"]]
    n_valid = len(valid)

    correct = [r for r in valid if r["correct"]]
    accuracy = len(correct) / n_valid if n_valid else 0

    print(f"=== {label} ===")
    print(f"Total examples: {n}")
    print(f"Valid JSON: {n_valid}/{n} ({100*n_valid/n:.1f}%)")
    print(f"Accuracy (on valid outputs): {len(correct)}/{n_valid} ({100*accuracy:.1f}%)")

    print("\nPer-class breakdown:")
    for action in ["buy", "sell", "hold"]:
        class_examples = [r for r in valid if r["true_action"] == action]
        class_correct = [r for r in class_examples if r["correct"]]
        n_class = len(class_examples)
        acc = len(class_correct)/n_class if n_class else 0
        pred_dist = {}
        for r in class_examples:
            pred_dist[r["pred_action"]] = pred_dist.get(r["pred_action"], 0) + 1
        print(f"  {action:5s}: {len(class_correct)}/{n_class} correct ({100*acc:.1f}%)  |  predicted as: {pred_dist}")

    print("\nConfidence calibration:")
    buckets = {"low (0-0.5)": [], "mid (0.5-0.7)": [], "high (0.7-1.0)": []}
    for r in valid:
        c = r["pred_confidence"] or 0
        if c < 0.5: buckets["low (0-0.5)"].append(r)
        elif c < 0.7: buckets["mid (0.5-0.7)"].append(r)
        else: buckets["high (0.7-1.0)"].append(r)
    for name, bucket in buckets.items():
        if bucket:
            acc = sum(r["correct"] for r in bucket)/len(bucket)
            print(f"  {name}: n={len(bucket)}, accuracy={100*acc:.1f}%")

    return {"n": n, "valid_pct": 100*n_valid/n, "accuracy": 100*accuracy}

ft_summary = summarize(ft_results, label="Fine-tuned Qwen1.5B")

=== Fine-tuned Qwen1.5B ===
Total examples: 50
Valid JSON: 50/50 (100.0%)
Accuracy (on valid outputs): 48/50 (96.0%)

Per-class breakdown:
  buy  : 19/19 correct (100.0%)  |  predicted as: {'buy': 19}
  sell : 15/15 correct (100.0%)  |  predicted as: {'sell': 15}
  hold : 14/16 correct (87.5%)  |  predicted as: {'hold': 14, 'sell': 1, 'buy': 1}

Confidence calibration:
  low (0-0.5): n=14, accuracy=100.0%
  mid (0.5-0.7): n=20, accuracy=95.0%
  high (0.7-1.0): n=16, accuracy=93.8%


In [ ]:
from google.colab import files
uploaded = files.upload()  # pick ood_eval.jsonl

ood_data = load_jsonl("ood_eval.jsonl")
ood_results = run_eval(model, tokenizer, ood_data, label="OOD test")
ood_summary = summarize(ood_results, label="Fine-tuned Qwen1.5B — out-of-template")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saving ood_eval.jsonl to ood_eval.jsonl


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Fine-tuned Qwen1.5B — out-of-template ===
Total examples: 10
Valid JSON: 10/10 (100.0%)
Accuracy (on valid outputs): 6/10 (60.0%)

Per-class breakdown:
  buy  : 1/3 correct (33.3%)  |  predicted as: {'sell': 1, 'hold': 1, 'buy': 1}
  sell : 2/2 correct (100.0%)  |  predicted as: {'sell': 2}
  hold : 3/5 correct (60.0%)  |  predicted as: {'hold': 3, 'sell': 2}

Confidence calibration:
  low (0-0.5): n=4, accuracy=75.0%
  mid (0.5-0.7): n=6, accuracy=50.0%


In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
base_ft_results = run_eval(base_model, base_tokenizer, eval_data, label="base")
base_summary_templated = summarize(base_ft_results, label="Base Qwen1.5B — templated eval")

base_ood_results = run_eval(base_model, base_tokenizer, ood_data, label="base-ood")
base_summary_ood = summarize(base_ood_results, label="Base Qwen1.5B — OOD eval")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Base Qwen1.5B — templated eval ===
Total examples: 50
Valid JSON: 44/50 (88.0%)
Accuracy (on valid outputs): 24/44 (54.5%)

Per-class breakdown:
  buy  : 12/16 correct (75.0%)  |  predicted as: {'buy': 12, 'BUY': 4}
  sell : 5/13 correct (38.5%)  |  predicted as: {'short': 4, 'sell': 5, 'buy': 4}
  hold : 7/15 correct (46.7%)  |  predicted as: {'hold': 7, 'sell': 3, 'buy': 4, 'BUY': 1}

Confidence calibration:
  mid (0.5-0.7): n=1, accuracy=100.0%
  high (0.7-1.0): n=43, accuracy=53.5%


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Base Qwen1.5B — OOD eval ===
Total examples: 10
Valid JSON: 10/10 (100.0%)
Accuracy (on valid outputs): 5/10 (50.0%)

Per-class breakdown:
  buy  : 2/3 correct (66.7%)  |  predicted as: {'sell': 1, 'buy': 2}
  sell : 1/2 correct (50.0%)  |  predicted as: {'buy': 1, 'sell': 1}
  hold : 2/5 correct (40.0%)  |  predicted as: {'hold': 2, 'short': 1, 'wait': 1, 'sell': 1}

Confidence calibration:
  low (0-0.5): n=1, accuracy=100.0%
  high (0.7-1.0): n=9, accuracy=44.4%
